## Simple MAE Fine-tuning for Classification

- Pretrain/evaluate MAE reconstructions and attention.
- Fine-tune a classifier head on AffectNet (7 classes).
- Transfer to FER2013 with label remapping.

Contents:
- Dataset and transforms
- MAE encoder/decoder and utilities
- Filtering and distribution plots
- Classifier head
- Reconstruction and attention visualization
- Fine-tuning on AffectNet
- Transfer learning to FER2013
- Evaluation utilities


In [1]:
import os
import copy
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, Subset, ConcatDataset
from torchvision import datasets
import torchvision.transforms as transforms

import timm
from timm.models.vision_transformer import VisionTransformer

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

### Dataset: AffectNet for Classification

- **Images**: RGB faces in `image_dir`.
- **Annotations**: `.npy` files in `annotation_dir` with keys like `{id}_aro`, `{id}_val`, `{id}_exp`.
- **Goal**: Provide images and labels (arousal, valence, expression) for downstream MAE fine-tuning and evaluation.

In [2]:
class AffectNetClassificationDataset(Dataset):
    """AffectNet classification dataset.

    Expects images in `image_dir` and `.npy` annotation files in `annotation_dir`.
    Annotations are read once into memory for fast lookup.
    Returns a tuple: (image_tensor, { 'arousal', 'valence', 'expression' }).
    """
    def __init__(self, image_dir, annotation_dir, transform=None):
        self.image_dir = image_dir
        self.annotation_dir = annotation_dir
        self.image_files = sorted(os.listdir(image_dir))
        self.transform = transform
        self.annotations = {}

        annotation_files = [file for file in os.listdir(annotation_dir) if file.endswith('.npy')]
        for file in tqdm(annotation_files, desc="Loading AffectNet annotation files"):
            key = file.split('.')[0]
            npy_path = os.path.join(annotation_dir, file)
            self.annotations[key] = np.load(npy_path, allow_pickle=True)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        img_id = os.path.splitext(img_name)[0]
        aro = self.annotations.get(f'{img_id}_aro', np.array([-1]))
        val = self.annotations.get(f'{img_id}_val', np.array([-1]))
        exp = self.annotations.get(f'{img_id}_exp', np.array([-1]))
        aro = aro.item() if aro.size == 1 else aro
        val = val.item() if val.size == 1 else val
        exp = exp.item() if exp.size == 1 else exp

        labels = {
            'arousal': aro,
            'valence': val,
            'expression': exp
        }

        return image, labels

### MAE Model: ViT encoder and lightweight decoder

- **Encoder**: ViT-B/16 created via `timm` and optionally loaded from a local checkpoint.
- **Decoder**: Small transformer used only for reconstruction during MAE pretraining/evaluation.
- **Masking**: Random patch masking with ratio `mask_ratio`.

In [5]:
def create_mae_model_from_timm(new_depth=6):
    """Create a ViT-B/16 encoder and load local weights.

    - Avoids any download by setting `pretrained=False`.
    - Loads a local checkpoint if present (keys with optional `module.` prefix).
    """
    encoder = timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        img_size=224,
        patch_size=16,
        embed_dim=768,
        depth=new_depth,
        num_heads=12,
        mlp_ratio=4.0
    )

    ckpt_path = '../models/vit_base_patch16_224.pth'
    state_dict = torch.load(ckpt_path, map_location='cpu')

    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

    missing_keys, unexpected_keys = encoder.load_state_dict(state_dict, strict=False)
    print("✅ Loaded encoder weights.")
    print(f"Missing keys: {len(missing_keys)} | Unexpected keys: {len(unexpected_keys)}")

    return encoder


class SimpleMAE(torch.nn.Module):
    """Minimal MAE composed of a ViT encoder + lightweight decoder.

    - `forward(imgs)` returns `(loss, pred_tokens, mask, ids_restore)`.
    - `mask_ratio` controls the fraction of patches masked during encoding.
    """
    def __init__(
        self,
        encoder,
        decoder_embed_dim=512,
        decoder_depth=8,
        decoder_num_heads=16,
        mask_ratio=0.75,
        norm_pix_loss=True
    ):
        super().__init__()
        self.encoder = encoder
        self.img_size = encoder.patch_embed.img_size[0]
        self.patch_size = encoder.patch_embed.patch_size[0]
        self.embed_dim = encoder.embed_dim
        self.num_patches = (self.img_size // self.patch_size) ** 2
        self.mask_token = torch.nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_embed = torch.nn.Linear(self.embed_dim, decoder_embed_dim, bias=True)
        self.decoder_pos_embed = torch.nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim))
        self.decoder_blocks = torch.nn.ModuleList([
            timm.models.vision_transformer.Block(
                decoder_embed_dim, decoder_num_heads, 4.0
            ) for _ in range(decoder_depth)
        ])

        self.decoder_norm = torch.nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = torch.nn.Linear(
            decoder_embed_dim, self.patch_size**2 * 3, bias=True
        )

        self.mask_ratio = mask_ratio
        self.norm_pix_loss = norm_pix_loss
        self.initialize_weights()

    def initialize_weights(self):
        torch.nn.init.normal_(self.mask_token, std=0.02)
        torch.nn.init.normal_(self.decoder_pos_embed, std=0.02)
        torch.nn.init.xavier_uniform_(self.decoder_embed.weight)
        torch.nn.init.zeros_(self.decoder_embed.bias)
        torch.nn.init.xavier_uniform_(self.decoder_pred.weight)
        torch.nn.init.zeros_(self.decoder_pred.bias)

    def random_masking(self, x, mask_ratio):
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(
            x, dim=1,
            index=ids_keep.unsqueeze(-1).repeat(1, 1, D)
        )
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio):
        patches = self.encoder.patch_embed(x)
        patches = patches + self.encoder.pos_embed[:, 1:, :]
        patches_masked, mask, ids_restore = self.random_masking(patches, mask_ratio)
        cls_token = self.encoder.cls_token + self.encoder.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(patches_masked.shape[0], -1, -1)
        x = torch.cat((cls_tokens, patches_masked), dim=1)
        for blk in self.encoder.blocks:
            x = blk(x)
        x = self.encoder.norm(x)

        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)
        mask_tokens = self.mask_token.repeat(
            x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1
        )
        x_ = x[:, 1:, :]
        x_ = torch.cat([x_, mask_tokens], dim=1)
        x_ = torch.gather(
            x_, dim=1,
            index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2])
        )
        x = torch.cat([x[:, :1, :], x_], dim=1)
        x = x + self.decoder_pos_embed
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        x = self.decoder_pred(x)
        x = x[:, 1:, :]

        return x

    def patchify(self, imgs):
        p = self.patch_size
        h = w = self.img_size // p
        x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(shape=(imgs.shape[0], h * w, p**2 * 3))

        return x

    def unpatchify(self, x):
        p = self.patch_size
        h = w = int(x.shape[1]**0.5)
        assert h * w == x.shape[1]
        x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(shape=(x.shape[0], 3, h * p, w * p))

        return imgs

    def forward_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1.e-6)**0.5
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()

        return loss

    def forward(self, imgs, mask_ratio=None):
        if mask_ratio is None:
            mask_ratio = self.mask_ratio
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask, ids_restore

### Data loading and filtering

- Load preprocessed AffectNet datasets and apply standard train/val transforms.
- Filter out class `7` (Contempt) to align with a 7-class setup.

In [ ]:
def fast_filter_not_class(dataset: AffectNetClassificationDataset, bad_class: int = 7):
	"""Filter out images whose expression equals `bad_class`.

	Uses only filename + preloaded annotations. No image I/O.
	"""
	valid_names = []
	filtered = 0
	for name in tqdm(dataset.image_files, desc='Fast filter by label'):
		img_id = os.path.splitext(name)[0]
		exp = dataset.annotations.get(f'{img_id}_exp', np.array([-1]))
		exp = int(exp.item()) if isinstance(exp, np.ndarray) and exp.size == 1 else int(exp)
		if 0 <= exp < 8 and exp != bad_class:
			valid_names.append(name)
		else:
			filtered += 1
	print(f"Filtered out {filtered}; kept {len(valid_names)}")
	dataset.image_files = valid_names
	return dataset

train_data = torch.load("../datasets/AffectNet/AffectNet-torch/train_dataset.pt", weights_only=False)
val_data = torch.load("../datasets/AffectNet/AffectNet-torch/test_dataset.pt", weights_only=False)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_data.transform = train_transform
val_data.transform = val_transform
train_data.image_dir = "../datasets/AffectNet/AffectNet-images/train_set/images"
val_data.image_dir = "../datasets/AffectNet/AffectNet-images/val_set/images"

# -------- Helper functions --------

def count_expressions(dataset):
    """Count per-class expression frequencies in a dataset."""
    counter = Counter()
    for img_name in dataset.image_files:
        img_id = os.path.splitext(img_name)[0]
        exp = dataset.annotations.get(f'{img_id}_exp', None)
        if exp is not None and exp.size == 1:
            counter[exp.item()] += 1
        else:
            counter[-1] += 1
    return counter


def plot_expression_pie(counter, title):
    """Plot a pie chart of expression distribution with labels and counts."""
    affectnet_classes = {
        "0": "Neutral",
        "1": "Happy",
        "2": "Sad",
        "3": "Surprise",
        "4": "Fear",
        "5": "Disgust",
        "6": "Anger",
        "7": "Contempt",
        "-1": "Unknown/Invalid"
    }
    expressions = sorted(counter.keys())
    counts = [counter[e] for e in expressions]

    fig, ax = plt.subplots(figsize=(6, 6))
    wedges, texts, autotexts = ax.pie(
        counts,
        labels=[affectnet_classes.get(e, f"Class {e}") for e in expressions],
        autopct='%1.1f%%',
        startangle=140,
        wedgeprops={'edgecolor': 'black'}
    )

    # Legend with class names + counts
    legend_labels = [
        f"{affectnet_classes.get(e, f'Class {e}')}: {counter[e]} samples"
        for e in expressions
    ]
    ax.legend(
        wedges, legend_labels,
        title="AffectNet Classes",
        loc="center left",
        bbox_to_anchor=(1, 0, 0.5, 1)
    )

    ax.set_title(title)
    ax.axis('equal')
    plt.tight_layout()
    plt.show()

# -----------------------------------

# --- Before filtering ---
print("Counting expressions before filtering...")
train_counter_before = count_expressions(train_data)
plot_expression_pie(train_counter_before, "Train Set Before Filtering")

# --- Filter ---
print(f"Training data original length: {len(train_data)}")
print(f"Validation data original length: {len(val_data)}")

train_data = fast_filter_not_class(train_data)
val_data = fast_filter_not_class(val_data)

# --- After filtering ---
print("Counting expressions after filtering...")
train_counter_after = count_expressions(train_data)
val_counter_after = count_expressions(val_data)

plot_expression_pie(train_counter_after, "Train Set After Filtering")
plot_expression_pie(val_counter_after, "Validation Set After Filtering")


### Classifier head (FERClassifier)

- Uses the MAE encoder as a feature extractor.
- Applies `LayerNorm` + `Linear` on the `[CLS]` token for 7-way classification.

In [6]:
class FERClassifier(nn.Module):
    """Simple classifier head on top of a ViT-based MAE encoder.

    Uses the `[CLS]` token from the encoder as the global image representation.
    """
    def __init__(self, mae_encoder, num_classes=7):
        super().__init__()
        self.encoder = mae_encoder
        self.classifier = nn.Sequential(
            nn.LayerNorm(mae_encoder.embed_dim),
            nn.Linear(mae_encoder.embed_dim, num_classes)
        )

    def forward(self, x):
        tokens = self.encoder.patch_embed(x)
        cls_token = self.encoder.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_token, tokens), dim=1)
        x = x + self.encoder.pos_embed[:, :x.size(1), :]
        x = self.encoder.pos_drop(x)

        for blk in self.encoder.blocks:
            x = blk(x)

        x = self.encoder.norm(x)
        cls_embedding = x[:, 0]
        return self.classifier(cls_embedding)

### Load MAE and visualize reconstructions/attention

- Load MAE weights and move to device.
- Visualize original vs masked vs reconstruction.
- Hook the last attention layer to visualize attention maps.

In [7]:
# ========== 1. Load Model ==========
encoder = create_mae_model_from_timm(new_depth=6)
model = SimpleMAE(encoder)
model.load_state_dict(torch.load('../models/model_epoch_35.pth', map_location='cpu'))
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# ========== 2. Prepare DataLoader ==========
test_loader = DataLoader(val_data, batch_size=5, shuffle=True)

# ========== 3. Show Original, Masked, Reconstructed ==========
def show_image_comparison(original_imgs, masked_imgs, reconstructed_imgs):
    """Visualize original, masked, and reconstructed versions side-by-side."""
    original_imgs = original_imgs.cpu()
    masked_imgs = masked_imgs.cpu()
    reconstructed_imgs = reconstructed_imgs.cpu()
    fig, axes = plt.subplots(3, len(original_imgs), figsize=(15, 9))

    for i in range(len(original_imgs)):
        for row, img_tensor in enumerate([original_imgs[i], masked_imgs[i], reconstructed_imgs[i]]):
            img = img_tensor.permute(1, 2, 0).numpy()
            img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img = np.clip(img, 0, 1)
            axes[row, i].imshow(img)
            axes[row, i].axis('off')
            if row == 0:
                axes[row, i].set_title("Original")
            elif row == 1:
                axes[row, i].set_title("Masked")
            else:
                axes[row, i].set_title("Reconstruction")

    plt.tight_layout()
    plt.show()

# ========== 4. Patch Transformer to Expose Attention ==========
attn_maps = []

# Modify the attention module to store attention weights
last_block = model.encoder.blocks[-1]
attn_module = last_block.attn
original_forward = attn_module.forward

def forward_with_attn_output(self, x, attn_mask=None):
    """Forward pass wrapper that stores attention weights for visualization."""
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
    q, k, v = qkv.permute(2, 0, 3, 1, 4)
    attn = (q @ k.transpose(-2, -1)) * self.scale
    attn = attn.softmax(dim=-1)
    self.attn_weights = attn.detach().cpu()
    attn_maps.append(self.attn_weights)
    attn = self.attn_drop(attn)
    x = (attn @ v).transpose(1, 2).reshape(B, N, C)
    x = self.proj(x)
    x = self.proj_drop(x)
    return x

attn_module.forward = forward_with_attn_output.__get__(attn_module, type(attn_module))

# ========== 5. Utility to Generate Masked Input ==========
def patchify(x, patch_size=16):
    B, C, H, W = x.shape
    assert H % patch_size == 0 and W % patch_size == 0
    h, w = H // patch_size, W // patch_size
    x = x.reshape(B, C, h, patch_size, w, patch_size)
    x = x.permute(0, 2, 4, 3, 5, 1).reshape(B, h * w, patch_size * patch_size * C)
    return x

def unpatchify(x, patch_size=16, img_size=224):
    B, N, D = x.shape
    h = w = int(np.sqrt(N))
    C = D // (patch_size * patch_size)
    x = x.reshape(B, h, w, patch_size, patch_size, C)
    x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, C, img_size, img_size)
    return x

def apply_mask(imgs, mask, ids_restore, patch_size=16):
    """Apply the MAE mask to input images for visualization purposes."""
    x = patchify(imgs, patch_size)
    mask_tokens = torch.zeros_like(x)
    x_masked = x * (1. - mask.unsqueeze(-1)) + mask_tokens * mask.unsqueeze(-1)
    x_restored = torch.gather(x_masked, dim=1, index=ids_restore.unsqueeze(-1).expand(-1, -1, x.size(-1)))
    return unpatchify(x_restored, patch_size=patch_size, img_size=imgs.shape[-1])

# ========== 6. Visualize Attention ==========
def visualize_attention(imgs, attn_map):
    """Overlay the `[CLS]` attention map on the original image."""
    B, C, H, W = imgs.shape
    attn = attn_map.mean(1)
    cls_attn = attn[:, 0, 1:]
    num_patches = cls_attn.shape[1]
    patch_size = int(np.sqrt(num_patches))

    for i in range(min(B, 5)):
        img = imgs[i].permute(1, 2, 0).cpu().numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)

        attn_map_img = cls_attn[i].reshape(patch_size, patch_size).unsqueeze(0)
        attn_map_img = torch.nn.functional.interpolate(attn_map_img.unsqueeze(0), size=(H, W), mode='bilinear')[0, 0].numpy()

        plt.figure(figsize=(6, 3))
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title("Original Image")

        plt.subplot(1, 2, 2)
        plt.imshow(img)
        plt.imshow(attn_map_img, cmap='jet', alpha=0.5)
        plt.axis('off')
        plt.title("Attention Overlay")
        plt.show()

# ========== 7. Run Model and Visualize ==========
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        loss, pred, mask, ids_restore = model(imgs)
        recon_imgs = model.unpatchify(pred)
        masked_imgs = apply_mask(imgs, mask, ids_restore)

        print(f"Loss: {loss.item():.4f}")
        show_image_comparison(imgs, masked_imgs, recon_imgs)
        visualize_attention(imgs, attn_maps[0])
        break

# ========== 8. Cleanup ==========
attn_module.forward = original_forward


✅ Loaded encoder weights.
Missing keys: 0 | Unexpected keys: 72


SimpleMAE(
  (encoder): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (norm): Identity()
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (ac

### Fine-tuning on AffectNet (7 classes)

- Freeze encoder initially, then unfreeze after a few epochs.
- Track train/val loss and accuracy; save best model and per-epoch checkpoints.

In [9]:
# train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=os.cpu_count() or 4, pin_memory=(device.type=='cuda'), persistent_workers=True)
# val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=os.cpu_count() or 4, pin_memory=(device.type=='cuda'), persistent_workers=True)
# # Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model setup
encoder = model.encoder  # Your pretrained MAE encoder
fer_model = FERClassifier(mae_encoder=encoder, num_classes=7).to(device)
print(sum(p.numel() for p in fer_model.parameters()))

fer_model.to(device)
Freeze encoder initially
def set_encoder_trainable(model, trainable):
    """Enable/disable gradient updates for the encoder parameters."""
    for param in model.encoder.parameters():
        param.requires_grad = trainable

set_encoder_trainable(fer_model, False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, fer_model.parameters()), lr=1e-4)
# optimizer.load_state_dict(checkout["optimizer_state_dict"])

# Checkpoint setup
best_val_loss = float('inf')
best_model_path = '../checkouts/fer_test/best_fer_model.pth'

# Create optimizer again
# optimizer = optim.Adam(filter(lambda p: p.requires_grad, fer_model.parameters()), lr=1e-4)



start_epoch = 0
best_val_loss = 0


# Training loop
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

# Training loop
num_epochs = 20
for epoch in range(start_epoch, num_epochs):
    if epoch >= 8:
        print("🔓 Unfreezing encoder")
        set_encoder_trainable(fer_model, True)
        optimizer = optim.Adam(fer_model.parameters(), lr=1e-5)  # Lower LR for fine-tuning

    fer_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        exp_labels = torch.tensor([int(x) for x in labels['expression']], dtype=torch.long)
        valid_mask = (exp_labels >= 0) & (exp_labels < 7)
        if valid_mask.sum() == 0:
            continue

        images = images[valid_mask].to(device)
        labels = exp_labels[valid_mask].to(device)

        optimizer.zero_grad()
        outputs = fer_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    avg_train_loss = running_loss / len(train_loader)
    print(f"✅ Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Train Acc={train_acc:.2f}%")

    # Validation
    fer_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            exp_labels = torch.tensor([int(x) for x in val_labels['expression']], dtype=torch.long)
            valid_mask = (exp_labels >= 0) & (exp_labels < 7)
            if valid_mask.sum() == 0:
                continue

            val_images = val_images[valid_mask].to(device)
            val_labels = exp_labels[valid_mask].to(device)

            val_outputs = fer_model(val_images)
            val_loss += criterion(val_outputs, val_labels).item()
            _, val_preds = val_outputs.max(1)
            val_total += val_labels.size(0)
            val_correct += val_preds.eq(val_labels).sum().item()

    val_acc = 100 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    print(f"🔍 Val Loss={avg_val_loss:.4f}, Val Acc={val_acc:.2f}%")

    # Save to history
    history["train_loss"].append(avg_train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(avg_val_loss)
    history["val_acc"].append(val_acc)

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': fer_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': avg_val_loss,
        }, best_model_path)
        print(f"💾 Best model saved at epoch {epoch+1} with val loss {avg_val_loss:.4f}\n")

    # Save epoch checkpoint
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': fer_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': avg_val_loss,
    }, f'../checkouts/fer_test/fer_model_epoch_{epoch+1}.pth')
    print(f"📝 Checkpoint saved: fer_model_epoch_{epoch+1}.pth\n")




44047343


FERClassifier(
  (encoder): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (norm): Identity()
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
         

### Transfer learning to FER2013

- Remap FER2013 labels to match desired order.
- Split train into train/val; reuse the same encoder + classifier head.

In [ ]:
# Set paths
encoder = model.encoder  # Your pretrained MAE encoder
fer_model = FERClassifier(mae_encoder=encoder, num_classes=7).to(device)
checkout = torch.load("../checkouts/fer_test/best_model.pth", map_location="cpu")
fer_model.load_state_dict(checkout["model_state_dict"])

fer_model.to(device)
train_dir = "../datasets/FER_2013_enhanced/FER_preprocessed_train"
test_dir = "../datasets/FER_2013_enhanced/preprocessed_test"
desired_class_order = [
    "neutral",   # 0
    "happy",     # 1
    "sad",       # 2
    "surprise",  # 3
    "fear",      # 4
    "disgust",   # 5
    "anger"      # 6
]
original_class_order = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
label_remap = {
    original_class_order.index('neutral'): desired_class_order.index('neutral'),
    original_class_order.index('happy'): desired_class_order.index('happy'),
    original_class_order.index('sad'): desired_class_order.index('sad'),
    original_class_order.index('surprise'): desired_class_order.index('surprise'),
    original_class_order.index('fear'): desired_class_order.index('fear'),
    original_class_order.index('disgust'): desired_class_order.index('disgust'),
    original_class_order.index('angry'): desired_class_order.index('anger')
}

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),  # random crop & scale
    transforms.RandomHorizontalFlip(p=0.5),               # flip horizontally
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # color augmentation
    transforms.RandomRotation(15),                        # small rotations
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # deterministic resize for val/test
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=val_test_transform)

def remap_labels(dataset):
    """Return a new dataset with labels remapped to `desired_class_order`."""
    # Create new list of samples with remapped labels
    new_samples = []
    for path, label in dataset.samples:
        new_label = label_remap[label]
        new_samples.append((path, new_label))

    # Create new dataset with remapped labels
    new_dataset = datasets.ImageFolder(root=dataset.root, transform=dataset.transform)
    new_dataset.samples = new_samples
    new_dataset.targets = [s[1] for s in new_samples]
    new_dataset.classes = desired_class_order
    new_dataset.class_to_idx = {cls: i for i, cls in enumerate(desired_class_order)}
    return new_dataset

# Apply label remapping
full_train_dataset = remap_labels(full_train_dataset)
test_dataset = remap_labels(test_dataset)

# Split training into train and validation
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])
val_dataset.dataset = remap_labels(val_dataset.dataset)
print("Classes:", test_dataset.classes)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=os.cpu_count() or 4, pin_memory=(device.type=='cuda'), persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=os.cpu_count() or 4, pin_memory=(device.type=='cuda'), persistent_workers=True)

def set_encoder_trainable(model, trainable):
    """Enable/disable gradient updates for the encoder parameters."""
    for param in model.encoder.parameters():
        param.requires_grad = trainable

set_encoder_trainable(fer_model, False)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, fer_model.parameters()), lr=1e-4)

# Checkpoint setup
best_val_loss = float('inf')
best_model_path = 'best_fer_model.pth'

# Create optimizer again
optimizer = optim.Adam(filter(lambda p: p.requires_grad, fer_model.parameters()), lr=1e-4)

In [ ]:
start_epoch = 0
num_epochs = 25
best_val_loss = float('inf')
label_mapping = {
    4: 0,
    3: 1,
    5: 2,
    6: 3,
    2: 4,
    1: 5,
    0: 6
}
optimizer = optim.Adam(fer_model.parameters(), lr=1e-5)
for epoch in range(start_epoch, num_epochs):
    if epoch == 6:
        set_encoder_trainable(fer_model, True)
    fer_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        # Create mask on CPU first
        valid_mask = (labels >= 0) & (labels < 7)
        if valid_mask.sum().item() == 0:
            continue  # Skip batch

        # Move to device
        images = images.to(device)
        labels = labels.to(device)

        # Apply mask
        images = images[valid_mask]
        labels = labels[valid_mask]

        optimizer.zero_grad()
        outputs = fer_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    print(f"✅ Epoch {epoch+1}: Train Loss={running_loss:.4f}, Train Acc={train_acc:.2f}%")

    # Validation phase
    fer_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            # Create mask on CPU first
            valid_mask = (val_labels >= 0) & (val_labels < 7)
            if valid_mask.sum().item() == 0:
                continue

            # Move to device
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)

            # Apply mask
            val_images = val_images[valid_mask]
            val_labels = val_labels[valid_mask]

            val_outputs = fer_model(val_images)
            val_loss += criterion(val_outputs, val_labels).item()
            _, val_preds = val_outputs.max(1)
            predicted_mapped = torch.tensor([label_mapping[p.item()] for p in val_preds],
                                          device=device)
            val_total += val_labels.size(0)
            val_correct += predicted_mapped.eq(val_labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"🔍 Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': fer_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
        }, '../checkouts/fer_test/best___model.pth')
        print(f"💾 Best model saved at epoch {epoch+1} with val loss {val_loss:.4f}\n")
    else:
        print("")

    # Save checkpoint
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': fer_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
        'val_acc': val_acc,
    }, f'../checkouts/fer_test/fer_model___epoch_{epoch+1}.pth')
    print(f"📝 Checkpoint saved: fer_model_epoch_{epoch+1}.pth\n")